# Generating AI Sample Using Prompts Based on the AI-Free Sample

## Notebook overview

This notebook documents the workflow used to create the **synthetic testing samples** for the research project. It starts from the cleaned AI-free corpus, generates prompt files, uses those prompt files to create AI-written samples, and then creates a humanized version of the AI-generated texts. The final cells compute word-count statistics so the generated corpora can be checked for scale and balance.

**Main workflow**
1. Generate prompt files from the trimmed AI-free sample.
2. Generate AI-written samples from those prompts.
3. Inspect the generated files and summarize their word counts.
4. Humanize the AI-written files with the external API.
5. Summarize the humanized corpus for later comparison.

## Generating Prompts Using AI-Free Sample

### Code block 1 — Generate prompt files from the AI-free sample

This block sets up the OpenAI client and Drive paths, reads the trimmed AI-free testing sample, and writes prompt files that will later be used to generate synthetic AI responses. It is the first generation stage of the testing-sample pipeline.

In [ ]:
!pip install -q openai

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
import re
import time
from pathlib import Path
from openai import OpenAI

# =========================
# SETTINGS
# =========================
os.environ["OPENAI_API_KEY"] = "${REDACTED_SECRET}"

client = OpenAI()

INPUT_FOLDER = "${PROJECT_DATA_PATH}"
OUTPUT_FOLDER = "${PROJECT_DATA_PATH}"

MODEL = "gpt-4.1-mini"
MAX_CHARS_PER_FILE = 45000   # increase if needed, but this helps control cost
SLEEP_BETWEEN_CALLS = 1.0    # seconds

# How tight you want the requested range to be around the real file length
RANGE_PERCENT = 0.10         # 10% on each side
MIN_RANGE_PADDING = 150      # minimum extra words on each side

input_path = Path(INPUT_FOLDER)
output_path = Path(OUTPUT_FOLDER)
output_path.mkdir(parents=True, exist_ok=True)

txt_files = sorted([p for p in input_path.iterdir() if p.is_file() and p.suffix.lower() == ".txt"])

if not txt_files:
    raise FileNotFoundError(f"No .txt files found in: {INPUT_FOLDER}")

def read_text_file(path):
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin-1"]:
        try:
            return path.read_text(encoding=enc)
        except:
            pass
    raise ValueError(f"Could not read file: {path.name}")

def count_words(text):
    # Counts words in a simple robust way
    return len(re.findall(r"\S+", text))

def get_word_range(word_count, pct=RANGE_PERCENT, min_padding=MIN_RANGE_PADDING):
    pad = max(int(round(word_count * pct)), min_padding)
    low = max(1, word_count - pad)
    high = word_count + pad
    return low, high

def build_messages(file_name, file_text):
    truncated_text = file_text[:MAX_CHARS_PER_FILE]
    word_count = count_words(file_text)
    low_wc, high_wc = get_word_range(word_count)

    system_msg = f"""
You are helping with a research workflow.

Task:
Read a student's finished project/report text file and infer ONE realistic prompt that the student could have asked ChatGPT so that ChatGPT could produce a document like this as the output.

Important requirements for the prompt you write:
1. The prompt must sound like a real student request to ChatGPT.
2. It must ask ChatGPT to write/complete the project or report, not to analyze it.
3. It must capture the likely task, deliverable, structure, tone, level, and major content requirements.
4. It should be specific enough that a model could generate something similar in genre, scope, and style.
5. The prompt MUST explicitly request a final output length in approximately the same range as the original file.
6. The target word-count range that must be requested in the prompt is: {low_wc} to {high_wc} words.
7. Make that word-count request sound natural, like something a student would actually ask for.
8. Do NOT quote or copy long passages from the file.
9. Do NOT mention that you were given the finished file.
10. Do NOT mention AI detection, reverse engineering, or research.
11. Output ONLY the prompt text, with no title, no bullets, no explanation, and no quotation marks unless naturally needed.
"""

    user_msg = f"""
File name: {file_name}
Actual file word count: {word_count}
Required target range to include in the inferred prompt: {low_wc} to {high_wc} words

Finished student file content:
--------------------
{truncated_text}
--------------------

Write the single best student-style prompt.
"""

    return [
        {"role": "system", "content": system_msg.strip()},
        {"role": "user", "content": user_msg.strip()},
    ]

success = 0
failed = []

for i, file_path in enumerate(txt_files, start=1):
    try:
        file_text = read_text_file(file_path).strip()

        if not file_text:
            print(f"[{i}/{len(txt_files)}] Skipped empty file: {file_path.name}")
            failed.append((file_path.name, "Empty file"))
            continue

        response = client.responses.create(
            model=MODEL,
            input=build_messages(file_path.name, file_text)
        )

        prompt_text = response.output_text.strip()

        if not prompt_text:
            raise ValueError("Model returned empty output.")

        out_file = output_path / file_path.name
        out_file.write_text(prompt_text, encoding="utf-8")

        print(f"[{i}/{len(txt_files)}] Saved prompt: {out_file.name}")
        success += 1
        time.sleep(SLEEP_BETWEEN_CALLS)

    except Exception as e:
        print(f"[{i}/{len(txt_files)}] FAILED: {file_path.name} -> {e}")
        failed.append((file_path.name, str(e)))

print("\n=========================")
print(f"Done. Saved {success} prompt files to:")
print(OUTPUT_FOLDER)

if failed:
    print("\nFailed files:")
    for name, err in failed:
        print(f"- {name}: {err}")
else:
    print("\nNo failures.")

### Code block 2 — Generate AI-written sample files from the prompt folder

This block reads the prompt files created in the previous stage and sends them to the model to generate full AI-written sample texts. The outputs are saved as the AI-generated testing sample used later in the comparison study.

In [ ]:
import os
!pip install -q openai

drive.mount('/content/drive', force_remount=False)


# =========================
# SETTINGS
# =========================
os.environ["OPENAI_API_KEY"] = "${REDACTED_SECRET}"

client = OpenAI()

PROMPTS_FOLDER = "${PROJECT_DATA_PATH}"
OUTPUT_FOLDER  = "${PROJECT_DATA_PATH}"

MODEL = "gpt-4.1-mini"
SLEEP_BETWEEN_CALLS = 1.0

prompts_path = Path(PROMPTS_FOLDER)
output_path = Path(OUTPUT_FOLDER)
output_path.mkdir(parents=True, exist_ok=True)

prompt_files = sorted([p for p in prompts_path.iterdir() if p.is_file() and p.suffix.lower() == ".txt"])

if not prompt_files:
    raise FileNotFoundError(f"No .txt files found in: {PROMPTS_FOLDER}")

def read_text_file(path):
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin-1"]:
        try:
            return path.read_text(encoding=enc)
        except:
            pass
    raise ValueError(f"Could not read file: {path.name}")

def clean_model_output(text):
    text = text.strip()

    # Remove markdown fences if model wraps output in them
    text = re.sub(r"^\s*```[a-zA-Z0-9_-]*\s*", "", text)
    text = re.sub(r"\s*```\s*$", "", text)

    # Remove common intro lines at the beginning
    intro_patterns = [
        r"^\s*here(?:'s| is)\b.*?\n+",
        r"^\s*certainly\b.*?\n+",
        r"^\s*sure\b.*?\n+",
        r"^\s*yes\b.*?\n+",
        r"^\s*i can help\b.*?\n+",
        r"^\s*below is\b.*?\n+",
        r"^\s*let(?: me)?(?:'s| us)?\b.*?\n+",
        r"^\s*of course\b.*?\n+",
    ]
    changed = True
    while changed:
        changed = False
        for pat in intro_patterns:
            new_text = re.sub(pat, "", text, flags=re.IGNORECASE | re.DOTALL)
            if new_text != text:
                text = new_text.strip()
                changed = True

    # Remove common ending lines at the end
    outro_patterns = [
        r"\n+\s*let me know if .*?$",
        r"\n+\s*if you(?:'d| would) like .*?$",
        r"\n+\s*feel free to .*?$",
        r"\n+\s*i can also .*?$",
        r"\n+\s*hope this helps.*?$",
        r"\n+\s*let me know whether .*?$",
        r"\n+\s*let me know if you need .*?$",
        r"\n+\s*you can ask me .*?$",
    ]
    changed = True
    while changed:
        changed = False
        for pat in outro_patterns:
            new_text = re.sub(pat, "", text, flags=re.IGNORECASE | re.DOTALL)
            if new_text != text:
                text = new_text.strip()
                changed = True

    return text.strip()

success = 0
failed = []

SYSTEM_INSTRUCTION = """
You are writing the final deliverable only.

Return only the requested project/report/assignment content itself.
Do NOT add any introduction, explanation, acknowledgment, preface, summary note, or closing note.
Do NOT say things like:
- "Sure, here is..."
- "I can help with that"
- "Let me know if you want changes"
- "Hope this helps"

Output only the final document text.
"""

for i, file_path in enumerate(prompt_files, start=1):
    try:
        prompt_text = read_text_file(file_path).strip()

        if not prompt_text:
            print(f"[{i}/{len(prompt_files)}] Skipped empty prompt file: {file_path.name}")
            failed.append((file_path.name, "Empty prompt file"))
            continue

        response = client.responses.create(
            model=MODEL,
            input=[
                {"role": "system", "content": SYSTEM_INSTRUCTION.strip()},
                {"role": "user", "content": prompt_text}
            ]
        )

        generated_text = response.output_text.strip()

        if not generated_text:
            raise ValueError("Model returned empty output.")

        generated_text = clean_model_output(generated_text)

        if not generated_text:
            raise ValueError("Output became empty after cleanup.")

        out_file = output_path / file_path.name
        out_file.write_text(generated_text, encoding="utf-8")

        success += 1
        print(f"[{i}/{len(prompt_files)}] Saved generated file: {out_file.name}")

        time.sleep(SLEEP_BETWEEN_CALLS)

    except Exception as e:
        print(f"[{i}/{len(prompt_files)}] FAILED: {file_path.name} -> {e}")
        failed.append((file_path.name, str(e)))

print("\n=========================")
print(f"Done. Saved {success} generated files to:")
print(OUTPUT_FOLDER)

if failed:
    print("\nFailed files:")
    for name, err in failed:
        print(f"- {name}: {err}")
else:
    print("\nNo failures.")

### Code block 3 — Inspect generated files and print file-level previews

This block opens the generated TXT files from Drive, checks that the files were created correctly, and prints quick previews or counts for manual inspection. It serves as a basic integrity check before later processing.

In [ ]:
drive.mount("/content/drive", force_remount=False)


# ====== CHANGE THIS ======
FOLDER = "${PROJECT_DATA_PATH}"
# =========================

folder_path = Path(FOLDER)

txt_files = sorted(
    [p for p in folder_path.iterdir() if p.is_file() and p.suffix.lower() == ".txt"]
)

if not txt_files:
    raise FileNotFoundError(f"No .txt files found in: {FOLDER}")

for file_path in txt_files:
    for enc in ["utf-8", "utf-8-sig", "cp1252", "latin-1"]:
        try:
            text = file_path.read_text(encoding=enc)
            break
        except:
            text = None
    if text is None:
        print(f"Could not read: {file_path.name}")
        continue

    cleaned = text.replace("###", "").replace("---", "").replace("**", "")
    file_path.write_text(cleaned, encoding="utf-8")
    print(f"Cleaned: {file_path.name}")

print("Done.")

## Statistics

### Code block 4 — Summarize the generated AI sample

This block computes word-count statistics for the generated AI sample. The summary helps verify corpus size, detect extreme files, and compare the generated sample against the other testing-sample branches.

In [ ]:
drive.mount("/content/drive", force_remount=False)

import os
import re
from collections import defaultdict

ROOT = "${PROJECT_DATA_PATH}"
BIN = 600


def count_words(text: str) -> int:
    return len(re.findall(r"[\w\u0600-\u06FF]+(?:['-][\w\u0600-\u06FF]+)?", text))


def bin_label(w: int) -> str:
    if w < BIN:
        return f"<{BIN}"
    lo = (w // BIN) * BIN
    hi = lo + BIN
    return f"{lo}-{hi}"


file_rows = []  # (subfolder, relpath, words)
total_words_all = 0

for root, _, files in os.walk(ROOT):
    for fn in files:
        if not fn.lower().endswith(".txt"):
            continue
        path = os.path.join(root, fn)
        rel = os.path.relpath(path, ROOT)
        subfolder = os.path.dirname(rel) if os.path.dirname(rel) else "."
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            txt = f.read()
        wc = count_words(txt)
        total_words_all += wc
        file_rows.append((subfolder, rel, wc))

if not file_rows:
    print("No .txt files found under:", ROOT)
else:
    # ---- files <300 ----
    lt300 = sorted(
        [r for r in file_rows if r[2] < 300], key=lambda x: (x[0], x[2], x[1])
    )
    print("FILES <300 WORDS (subfolder | words | file):")
    for sub, rel, wc in lt300:
        print(f"{sub} | {wc} | {rel}")

    # ---- per-subfolder bin distribution + per-subfolder word totals ----
    per_sub_counts = defaultdict(lambda: defaultdict(int))
    per_sub_files = defaultdict(int)
    per_sub_words = defaultdict(int)

    max_words = max(wc for _, _, wc in file_rows)
    labels = [f"<{BIN}"]
    hi = BIN
    while hi <= ((max_words // BIN) + 2) * BIN:
        labels.append(f"{hi}-{hi+BIN}")
        hi += BIN

    for sub, _, wc in file_rows:
        per_sub_files[sub] += 1
        per_sub_words[sub] += wc
        per_sub_counts[sub][bin_label(wc)] += 1

    print("\nPER-SUBFOLDER DISTRIBUTION (600-word bins):")
    for sub in sorted(per_sub_files.keys()):
        n = per_sub_files[sub]
        print(f"\n{sub}  (total files: {n} | total words: {per_sub_words[sub]})")
        for lab in labels:
            c = per_sub_counts[sub].get(lab, 0)
            pct = 100.0 * c / n
            if pct == 0:
                continue
            print(f"{lab}: {c} files ({pct:.2f}%)")

    # ---- word count per subfolder (requested) ----
    print(
        "\nWORD COUNT PER SUBFOLDER (subfolder | total words | files | avg words/file):"
    )
    for sub in sorted(per_sub_files.keys()):
        n = per_sub_files[sub]
        tw = per_sub_words[sub]
        avg = 0.0 if n == 0 else (tw / n)
        print(f"{sub} | {tw//1000}K | {n} | {(100*(avg//100)):.0f}")

    # ---- total words overall ----
    print("\nTOTAL WORDS ACROSS ALL FILES:", total_words_all)

# Generating Humanized Sample Using the AI-Generated Sample

Note: Below is the latest attempt. The first attempts finished processing a number files and had some time-out failures.

### Code block 5 — Create the humanized version of the AI-generated sample

This block sends the AI-generated texts to the external humanization service, stores the humanized outputs in a separate folder, and tracks progress so interrupted runs can be resumed safely.

In [ ]:
import json
import os
import re
import time
from pathlib import Path

import requests

INPUT_FOLDER = os.getenv("INPUT_FOLDER", "data/input")
OUTPUT_FOLDER = os.getenv("OUTPUT_FOLDER", "data/output")
PROGRESS_FOLDER = os.getenv("PROGRESS_FOLDER", "data/progress")

EMAIL = os.getenv("CONTACT_EMAIL")
PASSWORD = os.getenv("API_PASSWORD")
API_URL = os.getenv("HUMANIZER_API_URL", "https://ai-text-humanizer.com/api.php")

MAX_WORDS_PER_REQUEST = 1500
CONNECT_TIMEOUT = 20
READ_TIMEOUT = 300
SLEEP_BETWEEN_REQUESTS = 2.0
RETRIES_PER_CHUNK = 3
BACKOFF_SECONDS = [10, 20, 40]


def count_words(text: str) -> int:
    return len(re.findall(r"\S+", text))


def split_into_paragraphs(text: str):
    text = text.replace("\r\n", "\n").replace("\r", "\n").strip()
    if not text:
        return []

    paragraphs = re.split(r"\n\s*\n+", text)
    return [paragraph.strip() for paragraph in paragraphs if paragraph.strip()]


def split_long_paragraph(paragraph: str, max_words: int):
    words = re.findall(r"\S+", paragraph)
    if len(words) <= max_words:
        return [paragraph]

    # Prefer sentence boundaries; split oversized sentences by word count.
    sentences = re.split(r"(?<=[.!?])\s+", paragraph.strip())
    chunks = []
    current = []

    for sentence in sentences:
        sentence_words = count_words(sentence)
        current_words = count_words(" ".join(current)) if current else 0

        if sentence_words > max_words:
            sentence_word_list = re.findall(r"\S+", sentence)
            start = 0
            while start < len(sentence_word_list):
                part = " ".join(sentence_word_list[start : start + max_words])
                if current:
                    chunks.append(" ".join(current).strip())
                    current = []
                chunks.append(part.strip())
                start += max_words
            continue

        if current_words + sentence_words <= max_words:
            current.append(sentence)
        else:
            if current:
                chunks.append(" ".join(current).strip())
            current = [sentence]

    if current:
        chunks.append(" ".join(current).strip())

    return [chunk for chunk in chunks if chunk.strip()]


def chunk_text_by_paragraphs(text: str, max_words: int = MAX_WORDS_PER_REQUEST):
    paragraphs = split_into_paragraphs(text)
    if not paragraphs:
        return []

    chunks = []
    current_chunk_parts = []
    current_word_count = 0

    for paragraph in paragraphs:
        paragraph_word_count = count_words(paragraph)

        if paragraph_word_count > max_words:
            if current_chunk_parts:
                chunks.append("\n\n".join(current_chunk_parts).strip())
                current_chunk_parts = []
                current_word_count = 0

            chunks.extend(split_long_paragraph(paragraph, max_words))
            continue

        if current_word_count + paragraph_word_count <= max_words:
            current_chunk_parts.append(paragraph)
            current_word_count += paragraph_word_count
        else:
            chunks.append("\n\n".join(current_chunk_parts).strip())
            current_chunk_parts = [paragraph]
            current_word_count = paragraph_word_count

    if current_chunk_parts:
        chunks.append("\n\n".join(current_chunk_parts).strip())

    return chunks


def call_api(text: str):
    response = requests.post(
        API_URL,
        data={"email": EMAIL, "pw": PASSWORD, "text": text},
        timeout=(CONNECT_TIMEOUT, READ_TIMEOUT),
    )

    if response.status_code == 200:
        return response.text

    raise RuntimeError(
        f"API error: status_code={response.status_code}, response={response.text[:500]}"
    )


def file_already_completed(output_path: Path) -> bool:
    return (
        output_path.exists()
        and output_path.is_file()
        and output_path.stat().st_size > 0
    )


def get_progress_path(input_path: Path) -> Path:
    progress_dir = Path(PROGRESS_FOLDER)
    progress_dir.mkdir(parents=True, exist_ok=True)
    return progress_dir / f"{input_path.stem}.json"


def load_progress(progress_path: Path):
    if progress_path.exists():
        try:
            with open(progress_path, "r", encoding="utf-8") as file:
                return json.load(file)
        except Exception:
            pass

    return {"completed_chunks": {}, "total_chunks": 0}


def save_progress(progress_path: Path, progress_data: dict):
    progress_path.parent.mkdir(parents=True, exist_ok=True)
    with open(progress_path, "w", encoding="utf-8") as file:
        json.dump(progress_data, file, ensure_ascii=False, indent=2)


def finalize_output_from_progress(progress_data: dict, output_path: Path):
    completed_chunks = progress_data.get("completed_chunks", {})
    if not completed_chunks:
        return

    ordered_keys = sorted(completed_chunks.keys(), key=int)
    final_output = "\n\n".join(
        completed_chunks[key].strip() for key in ordered_keys
    ).strip()

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w", encoding="utf-8") as file:
        file.write(final_output)


def process_file(input_path: Path, output_path: Path):
    with open(input_path, "r", encoding="utf-8", errors="ignore") as file:
        original_text = file.read()

    chunks = chunk_text_by_paragraphs(original_text)

    if not chunks:
        print(f"Skipping empty file: {input_path.name}")
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, "w", encoding="utf-8") as file:
            file.write("")
        return

    progress_path = get_progress_path(input_path)
    progress = load_progress(progress_path)
    progress["total_chunks"] = len(chunks)

    print(f"\nProcessing: {input_path.name}")
    print(f"Total chunks: {len(chunks)}")

    for index, chunk in enumerate(chunks, start=1):
        chunk_key = str(index)

        if chunk_key in progress["completed_chunks"]:
            print(f"  Chunk {index}/{len(chunks)} already done, skipping.")
            continue

        print(f"  Chunk {index}/{len(chunks)} | words={count_words(chunk)}")
        success = False
        last_error = None

        for attempt in range(1, RETRIES_PER_CHUNK + 1):
            try:
                output_text = call_api(chunk)
                progress["completed_chunks"][chunk_key] = output_text.strip()
                save_progress(progress_path, progress)
                success = True
                break
            except Exception as error:
                last_error = error
                print(f"    Attempt {attempt} failed: {error}")

                if attempt < RETRIES_PER_CHUNK:
                    wait_time = BACKOFF_SECONDS[
                        min(attempt - 1, len(BACKOFF_SECONDS) - 1)
                    ]
                    print(f"    Waiting {wait_time} seconds before retry...")
                    time.sleep(wait_time)

        if not success:
            raise RuntimeError(
                f"Failed on file '{input_path.name}', chunk {index}: {last_error}"
            )

        time.sleep(SLEEP_BETWEEN_REQUESTS)

    finalize_output_from_progress(progress, output_path)
    print(f"Saved: {output_path}")


def main():
    if not EMAIL or not PASSWORD:
        raise ValueError("Set CONTACT_EMAIL and API_PASSWORD environment variables.")

    input_folder = Path(INPUT_FOLDER)
    output_folder = Path(OUTPUT_FOLDER)

    if input_folder.resolve() == output_folder.resolve():
        raise ValueError(
            "INPUT_FOLDER and OUTPUT_FOLDER must be different directories."
        )

    if not input_folder.exists():
        raise FileNotFoundError(f"Input folder does not exist: {input_folder}")

    txt_files = sorted(input_folder.glob("*.txt"))
    if not txt_files:
        raise FileNotFoundError(f"No .txt files found in: {input_folder}")

    output_folder.mkdir(parents=True, exist_ok=True)
    Path(PROGRESS_FOLDER).mkdir(parents=True, exist_ok=True)

    print(f"Found {len(txt_files)} txt files.")
    print(f"Input folder : {input_folder}")
    print(f"Output folder: {output_folder}")
    print(f"Progress dir : {PROGRESS_FOLDER}")

    skipped_files = []
    failed_files = []

    for txt_file in txt_files:
        try:
            output_path = output_folder / txt_file.name

            if file_already_completed(output_path):
                print(f"Skipping already completed: {txt_file.name}")
                skipped_files.append(txt_file.name)
                continue

            process_file(txt_file, output_path)
        except Exception as error:
            print(f"FAILED: {txt_file.name} -> {error}")
            failed_files.append((txt_file.name, str(error)))

    print("\n=========================")
    print("DONE")
    print("=========================")
    print(f"Skipped already done : {len(skipped_files)}")
    print(
        f"Processed this run   : {len(txt_files) - len(skipped_files) - len(failed_files)}"
    )
    print(f"Failed               : {len(failed_files)}")

    if skipped_files:
        print("\nSkipped files:")
        for name in skipped_files:
            print(f"- {name}")

    if failed_files:
        print("\nFailed files:")
        for name, error in failed_files:
            print(f"- {name}: {error}")


if __name__ == "__main__":
    main()

## Statistics

### Code block 6 — Summarize the humanized sample

This block computes the same word-count style statistics for the humanized sample so its size and distribution can be compared against the AI-generated and AI-free samples.